# Create Alfred P. Sloan Foundation Grant Awards (GRANT PATTERN, Method 6 Playwright)

The Alfred P. Sloan Foundation is a US philanthropic funder supporting research and education in science, technology, engineering, mathematics, and economics. This ingest covers the foundation's **public grants database** (`sloan.org/grants-database`), one row per grant.

**Method 6 (Playwright).** sloan.org sits behind a Cloudflare JS challenge that 403s plain HTTP requests. The scraper (`scripts/local/sloan_to_s3.py`) uses Playwright + playwright-stealth via the shared `scripts/local/_playwright_helper.py`. Sloan's challenge is heavier than Klingenstein's: it clears on the **first** navigation of a fresh browser context, then hard-sticks the challenge on any reuse — so the scraper recreates the browser context per page (`PlaywrightSession.recreate_context()`).

**Awarding body:** Alfred P. Sloan Foundation - F4320306151 (US, DOI 10.13039/100000879)

**Source:** `sloan.org/grants-database?page=N` (10 grants/page). Each grant card carries grantee org, amount, city, year, a brief description, Program / Sub-program / Initiative tags, an Investigator, and a permalink `/grant-detail/g-{YEAR}-{ID}`.

**Coverage note:** the online grants database only goes back to **2008**. Pre-2008 Sloan grants are not in this source (they require IRS 990-PF parsing — tracked separately). This ingest therefore covers 2008-present.

**Schema choices:**
- One row per grant. `funder_award_id` = the source permalink slug, e.g. `g-2025-79300` (stable, unique, source-authoritative).
- `funding_type` = `grant` uniformly.
- `funder_scheme` = the grant's top-level **Program** (Research, Higher Education, Technology, Public Understanding of Science and Technology, New York City Program, ...). Sub-program and Initiative are finer-grained tags retained in the raw staging table for QA but not surfaced as schemes (keeping the scheme space at the program granularity Sloan itself uses).
- **Amount IS published per grant** (`$25,000` form) → `amount` (DOUBLE) + `currency` = `USD`. **§6.7 is NOT waived** — Sloan posts a real dollar figure on every grant card.
- `lead_investigator` = the named Investigator (person) when present, else an org-only lead. `affiliation.name` = the grantee organization in both cases. `affiliation.country` is derived from the grantee city/region: `US` when the region is a US state/territory code, else mapped from a small country table, else NULL (never guessed).
- `description` = the grant's brief description (well populated on the source).

**S3 location:** `s3a://openalex-ingest/awards/sloan/sloan_grants.parquet`

**Provenance:** `sloan_foundation` (verified count=0 on production 2026-05-29).

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sloan_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/sloan/sloan_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.sloan_raw;

## Step 1.5: Inspect raw + program/year/coverage

Expected: ~3,419 grants, 2008-2026. Amount and grantee_org near-100%; country high (most grantees US, international grantees mapped where possible).

In [ ]:
%sql
DESCRIBE openalex.awards.sloan_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.sloan_raw LIMIT 5;

In [ ]:
%sql
-- Per-(program, year) row counts + coverage.
SELECT program, year, COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       COUNT(investigator) AS has_investigator,
       COUNT(country) AS has_country
FROM openalex.awards.sloan_raw
GROUP BY program, year
ORDER BY year DESC, rows DESC;

In [ ]:
%sql
-- Top grantee organizations (sanity check parsing didn't bleed fields).
SELECT grantee_org, COUNT(*) AS rows
FROM openalex.awards.sloan_raw
WHERE grantee_org IS NOT NULL
GROUP BY grantee_org
ORDER BY rows DESC
LIMIT 15;

## Step 1.6: Fail-fast - verify Alfred P. Sloan Foundation funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306151;

## Step 2: Transform to award schema

Per-row `display_name` = `{program} - {grantee_org} ({year})`. `lead_investigator` is the named Investigator (person) when present, else an org-only lead; `affiliation.name` = grantee org and `affiliation.country` is the parsed country in both cases.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sloan_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306151  -- Alfred P. Sloan Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT(COALESCE(n.program, 'Sloan Grant'), ' - ', n.grantee_org,
           CASE WHEN n.year IS NOT NULL THEN CONCAT(' (', n.year, ')') ELSE '' END) AS display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    n.program AS funder_scheme,
    'sloan_foundation' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.year AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.grantee_org IS NOT NULL OR n.investigator IS NOT NULL THEN struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.grantee_org AS name,
                n.country AS country,  -- parsed from grantee city/region; NULL when unmappable (never guessed)
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.permalink_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.sloan_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.grantee_org IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 149

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'sloan_foundation' AND priority = 149;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    149 AS priority  -- Alfred P. Sloan Foundation grants
FROM openalex.awards.sloan_awards;

## Step 6: Verification

§6.7 amount-coverage check **applies** (NOT waived): Sloan publishes a dollar figure per grant, so `pct_amount` should be near 100%. Other completeness checks: grantee_org / display_name near-100%, description high, start_year 100%.

In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.sloan_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.sloan_awards;

In [ ]:
%sql
-- §6.3 Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_pi,
    COUNT(start_year) AS has_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.sloan_awards;

In [ ]:
%sql
-- §6.7 amount check (NOT waived). Expect pct_amount ~100% and currency = [USD].
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.sloan_awards;

In [ ]:
%sql
-- Program (scheme) split.
SELECT funder_scheme, COUNT(*) AS rows,
       MIN(start_year) AS min_year, MAX(start_year) AS max_year,
       ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.sloan_awards
GROUP BY funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.sloan_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       funder_scheme AS program, funding_type, start_year, amount, currency,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       lead_investigator.affiliation.country AS country
FROM openalex.awards.sloan_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;